# MedFlow — Notebook 1 · Engenharia de Dados

**Sprint 2 · Ômega Urban Tech · Turma 1TSCOA**

Transforma os dados brutos do DATASUS em **bases curadas, simples e orientadas ao
que o MedFlow precisa medir**. Não calcula os índices — isso é o notebook 2.

## Contrato

| | |
|---|---|
| **Entrada** | `dados/processados/sih_sp_2022_2023_raw.parquet` (5.210.357 × 115)<br>`dados/processados/cnes_lt_sp_2022_2023_raw.parquet` (200.075 × 30)<br>`dados/referencias/municipios_ibge.csv` (645 municípios) |
| **Pré-requisito** | `00_extracao_dados.ipynb` executado |
| **Saída** | `dados/curados/` — 4 dimensões + 1 fato + 3 bases analíticas |
| **Recorte** | São Paulo, competências 2022-01 a 2023-12 |

## Princípio de segurança

Este notebook **nunca escreve em `dados/processados/`**. Ele grava em
`dados/curados/`, um diretório novo. Os parquets de IPH que sustentam os números
do pitch da Sprint 1 ficam intactos, e há cópia em `dados/_backup_parquets_originais/`.

## As três bases analíticas e por que existem

Cada base existe porque um índice precisa dela num grão específico:

| Base | Grão | Alimenta |
|---|---|---|
| `base_hospital_mes` | CNES × ano × mês | **IPH** (pressão) e **IS** (sazonalidade) |
| `base_hospital_espec_mes` | CNES × especialidade × ano × mês | **TMH** (mortalidade) e **CMI** (custo) |
| `base_hospital_cid` | CNES × CID-10 | **IPR** (permanência relativa) |

Nenhuma delas guarda um índice pronto — guardam os **ingredientes** (paciente-dia,
leitos, óbitos, custo, permanência). A divisão final acontece no notebook 2, onde
é fácil de auditar e de mudar.

In [1]:
from pathlib import Path
import calendar
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

# --- Caminhos -------------------------------------------------------------
BASE      = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_RAW   = BASE / "dados" / "processados"     # somente leitura
DIR_OUT   = BASE / "dados" / "curados"         # tudo que este notebook escreve
DIR_OUT.mkdir(parents=True, exist_ok=True)

# --- Parâmetros de execução ----------------------------------------------
ANOS = (2022, 2023)

# Se False, a gravação aborta caso o arquivo de saída já exista.
# Serve para que uma re-execução distraída não substitua um resultado validado.
SOBRESCREVER = False

print("entrada :", DIR_RAW)
print("saída   :", DIR_OUT)
assert DIR_RAW.exists(), f"não encontrei {DIR_RAW}"

entrada : <repo>/02_oracle_medflow/sprint_2_em_andamento/dados/processados
saída   : <repo>/02_oracle_medflow/sprint_2_em_andamento/dados/curados


---
## 1 · Carga dos brutos

Do SIH carregamos **20 das 115 colunas**. As outras 95 (diagnósticos secundários,
CPF do gestor, rubricas contábeis, campos de remessa) não participam de nenhum dos
5 índices e custam memória. A lista abaixo é o escopo declarado do projeto —
se um índice novo precisar de uma coluna, ela entra aqui explicitamente.

In [2]:
COLS_SIH = [
    "CNES",        # hospital — chave de junção com o CNES/LT
    "MUNIC_MOV",   # município de internação (6 dígitos, padrão SIH)
    "MUNIC_RES",   # município de residência do paciente
    "ESPEC",       # especialidade do leito
    "QT_DIARIAS",  # diárias pagas -> numerador paciente-dia do IPH
    "DIAS_PERM",   # dias de permanência (DT_SAIDA - DT_INTER)
    "MORTE",       # 1 = óbito -> numerador do TMH
    "VAL_TOT",     # valor total da AIH -> numerador do CMI
    "DIAG_PRINC",  # CID-10 principal -> grão do IPR
    "DT_INTER", "DT_SAIDA",
    "IDADE", "COD_IDADE", "SEXO",
    "CAR_INT",     # caráter da internação (eletiva/urgência)
    "COMPLEX",     # complexidade (média/alta)
    "MARCA_UTI",   # marca de UTI da AIH
    "UTI_MES_TO",  # diárias de UTI no mês
    "_ano", "_mes",  # competência, derivada na extração
]

sih = pd.read_parquet(DIR_RAW / "sih_sp_2022_2023_raw.parquet", columns=COLS_SIH)
cnes = pd.read_parquet(DIR_RAW / "cnes_lt_sp_2022_2023_raw.parquet")

print(f"SIH  : {sih.shape[0]:>9,} linhas × {sih.shape[1]} colunas")
print(f"CNES : {cnes.shape[0]:>9,} linhas × {cnes.shape[1]} colunas")
print(f"\ncompetências SIH : {sorted(sih._ano.unique())} × {len(sih._mes.unique())} meses")
print(f"hospitais no SIH : {sih.CNES.nunique()}")
print(f"municípios no SIH: {sih.MUNIC_MOV.nunique()}")

SIH  : 5,210,357 linhas × 20 colunas
CNES :   200,075 linhas × 30 colunas

competências SIH : [np.int64(2022), np.int64(2023)] × 12 meses
hospitais no SIH : 669


municípios no SIH: 331


---
## 2 · Diagnóstico de qualidade

Rodamos o diagnóstico **antes** de transformar qualquer coisa, para que os números
de qualidade sejam auditáveis e não uma nota de rodapé. Cada item aqui vira uma
decisão explícita mais adiante — nada é descartado em silêncio.

In [3]:
diag = {}

diag["linhas_sih"]        = len(sih)
diag["linhas_cnes"]       = len(cnes)
diag["hospitais_sih"]     = sih.CNES.nunique()
diag["municipios_sih"]    = sih.MUNIC_MOV.nunique()

# --- Integridade referencial ---------------------------------------------
sem_match = set(sih.CNES.unique()) - set(cnes.CNES.unique())
diag["hospitais_sem_cnes"] = len(sem_match)

# --- Campos que distorcem médias se não forem tratados --------------------
diag["qt_diarias_zero"]   = int((sih.QT_DIARIAS == 0).sum())
diag["dias_perm_zero"]    = int((sih.DIAS_PERM == 0).sum())
diag["val_tot_zero"]      = int((sih.VAL_TOT == 0).sum())
diag["obito_sem_valor"]   = int(((sih.VAL_TOT == 0) & (sih.MORTE == 1)).sum())

# --- Formato de códigos ---------------------------------------------------
diag["munic_mov_6_digitos"] = int((sih.MUNIC_MOV.astype(str).str.len() == 6).all())
diag["regsaude_vazia"]      = int((cnes.REGSAUDE.astype(str).str.strip() == "").sum())
diag["regsaude_distintas"]  = cnes.REGSAUDE.nunique()

for k, v in diag.items():
    print(f"{k:<24} {v:>12,}" if isinstance(v, (int, np.integer)) else f"{k:<24} {v:>12}")

print("\nQT_DIARIAS == DIAS_PERM em %.1f%% das AIH" % ((sih.QT_DIARIAS == sih.DIAS_PERM).mean() * 100))
print("→ são campos diferentes: QT_DIARIAS é diária faturada, DIAS_PERM é intervalo de datas.")
print("  O IPH usa QT_DIARIAS (o que o SUS pagou como ocupação).")

linhas_sih                  5,210,357
linhas_cnes                   200,075
hospitais_sih                     669
municipios_sih                    331
hospitais_sem_cnes                  0
qt_diarias_zero               400,958
dias_perm_zero                701,796
val_tot_zero                    7,827
obito_sem_valor                    20
munic_mov_6_digitos                 1
regsaude_vazia                 45,379
regsaude_distintas                 82

QT_DIARIAS == DIAS_PERM em 72.0% das AIH
→ são campos diferentes: QT_DIARIAS é diária faturada, DIAS_PERM é intervalo de datas.
  O IPH usa QT_DIARIAS (o que o SUS pagou como ocupação).


### O que o diagnóstico exige de decisão

| Achado | Volume | Decisão adotada |
|---|---|---|
| `QT_DIARIAS = 0` | ~401 mil | **Marcado com flag, não removido.** Entra no IPH (ocupação zero é ocupação real) e sai das médias de permanência do IPR |
| `VAL_TOT = 0` com óbito | 20 | Marcado com flag; excluído do CMI |
| `MUNIC_MOV` com 6 dígitos | 100% | Gera-se a versão de 7 dígitos do IBGE para joins externos |
| `REGSAUDE` vazia ou livre | ~22% | Normalização explícita na seção 6 |
| Hospitais do SIH sem CNES | 0 | Nada a fazer — cobertura total |

A regra geral: **flag em vez de descarte.** Quem consome a base decide o filtro, e
o filtro fica visível no notebook 2, não escondido aqui.

---
## 3 · `dim_tempo`

Doze meses × dois anos. Guarda `dias_no_mes`, que é o denominador do IPH — deixá-lo
numa dimensão evita recalcular data por data e torna o cálculo do índice legível.

In [4]:
dim_tempo = pd.DataFrame(
    [(a, m) for a in ANOS for m in range(1, 13)],
    columns=["_ano", "_mes"],
)
dim_tempo["dias_no_mes"]  = [calendar.monthrange(a, m)[1] for a, m in zip(dim_tempo._ano, dim_tempo._mes)]
dim_tempo["competencia"]  = dim_tempo._ano.astype(str) + dim_tempo._mes.astype(str).str.zfill(2)
dim_tempo["data_ref"]     = pd.to_datetime(dim_tempo.competencia + "01", format="%Y%m%d")
dim_tempo["trimestre"]    = dim_tempo.data_ref.dt.quarter
dim_tempo["mes_nome"]     = dim_tempo.data_ref.dt.strftime("%b")
dim_tempo["rotulo"]       = dim_tempo.mes_nome + "/" + dim_tempo._ano.astype(str).str[2:]

print(dim_tempo.to_string(index=False))

 _ano  _mes  dias_no_mes competencia   data_ref  trimestre mes_nome rotulo
 2022     1           31      202201 2022-01-01          1      Jan Jan/22
 2022     2           28      202202 2022-02-01          1      Feb Feb/22
 2022     3           31      202203 2022-03-01          1      Mar Mar/22
 2022     4           30      202204 2022-04-01          2      Apr Apr/22
 2022     5           31      202205 2022-05-01          2      May May/22
 2022     6           30      202206 2022-06-01          2      Jun Jun/22
 2022     7           31      202207 2022-07-01          3      Jul Jul/22
 2022     8           31      202208 2022-08-01          3      Aug Aug/22
 2022     9           30      202209 2022-09-01          3      Sep Sep/22
 2022    10           31      202210 2022-10-01          4      Oct Oct/22
 2022    11           30      202211 2022-11-01          4      Nov Nov/22
 2022    12           31      202212 2022-12-01          4      Dec Dec/22
 2023     1           31 

---
## 4 · `dim_especialidade`

O SIH traz `ESPEC` como código de dois dígitos. O material da Sprint 1 rotulou
`04` como *UTI* e `09` como *Crônico* — rótulos de apresentação, escolhidos para
comunicar a ideia, sem pretensão de oficialidade.

**Decisão para a Sprint 2: adotar a tabela oficial do SIH/SUS.** Ela lê `04` como
*Crônicos* e `09` como *Hospital-dia (cirúrgico)*, e acrescenta `06` (Tisiologia) e
`08` (Reabilitação), que o mapeamento anterior deixava de fora.

Consequência prática: **`ESPEC` deixa de ser fonte para qualquer recorte de UTI.**
Não existe código de UTI nessa tabela. Para isso o SIH tem campos próprios —
`MARCA_UTI` e `UTI_MES_TO` — que viram a flag `fl_uti` na seção 9 e são a base
correta para o tema.

In [5]:
# Tabela oficial de especialidade do leito — SIH/SUS. É a referência do projeto.
DEPARA_ESPEC = {
    "01": "Cirurgia",     "02": "Obstetrícia",  "03": "Clínica médica",
    "04": "Crônicos",     "05": "Psiquiatria",  "06": "Tisiologia",
    "07": "Pediatria",    "08": "Reabilitação", "09": "Hospital-dia (cirúrgico)",
}

# Rótulos usados na apresentação da Sprint 1, preservados só para rastrear de onde
# vieram os números daquele material. NÃO alimentam as bases.
DEPARA_SPRINT1_HISTORICO = {
    "01": "Cirúrgica", "02": "Obstétrica", "03": "Clínica", "04": "UTI",
    "05": "Psiquiatria", "07": "Pediatria", "09": "Crônico",
}

depara = DEPARA_ESPEC

obs = sih.ESPEC.value_counts().rename_axis("ESPEC").reset_index(name="internacoes")
dim_especialidade = obs.assign(
    especialidade = obs.ESPEC.map(depara).fillna("Outras / não mapeada"),
    mapeada       = obs.ESPEC.isin(depara).astype(int),
)
dim_especialidade["pct"] = (dim_especialidade.internacoes / len(sih) * 100).round(3)

nao_map = dim_especialidade.loc[dim_especialidade.mapeada == 0, "internacoes"].sum()
print(dim_especialidade.to_string(index=False))
print(f"\nfora do de-para: {nao_map:,} internações ({nao_map/len(sih)*100:.2f}%)")
print("→ não somem: viram a categoria 'Outras / não mapeada' e continuam somando nos totais.")

print("\nRastreio da mudança de rótulo em relação à Sprint 1:")
for cod in sorted(set(DEPARA_ESPEC) & set(DEPARA_SPRINT1_HISTORICO)):
    antes, agora = DEPARA_SPRINT1_HISTORICO[cod], DEPARA_ESPEC[cod]
    if antes != agora:
        n = int(dim_especialidade.loc[dim_especialidade.ESPEC == cod, "internacoes"].sum())
        print(f"  ESPEC {cod}: '{antes}' (Sprint 1) → '{agora}' (oficial) · {n:,} internações")
print("  Para recortes de UTI use a flag fl_uti (MARCA_UTI / UTI_MES_TO), não o ESPEC.")

ESPEC  internacoes            especialidade  mapeada    pct
   03      1805543           Clínica médica        1 34.653
   01      1729962                 Cirurgia        1 33.202
   02       723912              Obstetrícia        1 13.894
   07       459001                Pediatria        1  8.809
   09       267239 Hospital-dia (cirúrgico)        1  5.129
   05       126383              Psiquiatria        1  2.426
   04        64610                 Crônicos        1  1.240
   12        10504     Outras / não mapeada        0  0.202
   87         7369     Outras / não mapeada        0  0.141
   06         6619               Tisiologia        1  0.127
   14         3503     Outras / não mapeada        0  0.067
   10         2052     Outras / não mapeada        0  0.039
   08         1991             Reabilitação        1  0.038
   13         1630     Outras / não mapeada        0  0.031
   17           36     Outras / não mapeada        0  0.001
   11            3     Outras / não mape

---
## 5 · `dim_cid`

`DIAG_PRINC` tem 9.212 códigos distintos — granularidade demais para um painel de
gestão. Agrupamos pelos **22 capítulos da CID-10**, que é a leitura que um Secretário
de Saúde usa ("respiratório", "circulatório"), preservando o código original para
quem quiser descer ao detalhe.

In [6]:
# Capítulos da CID-10: (romano, letra_ini, num_ini, letra_fim, num_fim, descrição)
CAPITULOS_CID = [
    ("I",     "A00", "B99", "Infecciosas e parasitárias"),
    ("II",    "C00", "D48", "Neoplasias"),
    ("III",   "D50", "D89", "Sangue e órgãos hematopoéticos"),
    ("IV",    "E00", "E90", "Endócrinas, nutricionais e metabólicas"),
    ("V",     "F00", "F99", "Transtornos mentais e comportamentais"),
    ("VI",    "G00", "G99", "Sistema nervoso"),
    ("VII",   "H00", "H59", "Olho e anexos"),
    ("VIII",  "H60", "H95", "Ouvido e apófise mastoide"),
    ("IX",    "I00", "I99", "Aparelho circulatório"),
    ("X",     "J00", "J99", "Aparelho respiratório"),
    ("XI",    "K00", "K93", "Aparelho digestivo"),
    ("XII",   "L00", "L99", "Pele e tecido subcutâneo"),
    ("XIII",  "M00", "M99", "Osteomuscular e tecido conjuntivo"),
    ("XIV",   "N00", "N99", "Aparelho geniturinário"),
    ("XV",    "O00", "O99", "Gravidez, parto e puerpério"),
    ("XVI",   "P00", "P96", "Afecções do período perinatal"),
    ("XVII",  "Q00", "Q99", "Malformações congênitas"),
    ("XVIII", "R00", "R99", "Sintomas e achados anormais"),
    ("XIX",   "S00", "T98", "Lesões e envenenamentos"),
    ("XX",    "V01", "Y98", "Causas externas"),
    ("XXI",   "Z00", "Z99", "Fatores que influenciam o estado de saúde"),
    ("XXII",  "U04", "U99", "Códigos para propósitos especiais"),
]

def capitulo_cid(cod):
    '''Mapeia um código CID-10 (ex.: 'J189') para o capítulo correspondente.'''
    if not isinstance(cod, str) or len(cod) < 3:
        return ("--", "Não classificado")
    chave = cod[:3].upper()
    for romano, ini, fim, desc in CAPITULOS_CID:
        if ini <= chave <= fim:
            return (romano, desc)
    return ("--", "Não classificado")

codigos = sih.DIAG_PRINC.value_counts().rename_axis("DIAG_PRINC").reset_index(name="internacoes")
cap = codigos.DIAG_PRINC.map(capitulo_cid)
dim_cid = codigos.assign(
    capitulo      = [c[0] for c in cap],
    capitulo_desc = [c[1] for c in cap],
    categoria     = codigos.DIAG_PRINC.str[:3],
)

print(f"códigos distintos: {len(dim_cid):,}")
print(f"não classificados: {(dim_cid.capitulo == '--').sum()}")
print("\ntop 10 capítulos por volume de internação:")
print(dim_cid.groupby(["capitulo", "capitulo_desc"], as_index=False)
             .internacoes.sum()
             .sort_values("internacoes", ascending=False)
             .head(10).to_string(index=False))

códigos distintos: 9,212
não classificados: 0

top 10 capítulos por volume de internação:
capitulo                             capitulo_desc  internacoes
      XV               Gravidez, parto e puerpério       777019
      XI                        Aparelho digestivo       575900
      IX                     Aparelho circulatório       559670
     XIX                   Lesões e envenenamentos       524756
       X                     Aparelho respiratório       486748
      II                                Neoplasias       426826
     XIV                    Aparelho geniturinário       415420
       I                Infecciosas e parasitárias       251572
       V     Transtornos mentais e comportamentais       172645
     XXI Fatores que influenciam o estado de saúde       152853


---
## 6 · Região de saúde — a normalização do `REGSAUDE`

Esta é a promessa feita no slide 12 da Sprint 1: *"códigos não decodificados
(legado / ruído de cadastro) serão tratados no ETL da Sprint 2"*.

O campo está genuinamente sujo. Nos brutos convivem:

- códigos numéricos de 1 a 4 dígitos — `1`, `105`, `0105`, `201`, `0201`;
- rótulos de texto livre — `DRS1`, `GSP`, `XVI`, `XVII`, `MC`, `I`, `R17`;
- **45.379 linhas em branco** (22,7%).

**Decisões, em ordem:**

1. **Numérico → `zfill(4)`.** `105` e `0105` são a mesma região grafada de dois
   jeitos; padronizar em 4 dígitos funde as duas. Reduz 82 valores brutos a 49 códigos.
2. **Texto livre → nulo + flag.** `DRS1` e `GSP` **não** viram `0DRS1` nem `00GSP`.
   Preencher com zeros ali fabricaria um código que não existe. Ficam nulos e
   contabilizados.
3. **Fallback por município.** Um hospital sem região herda a região **modal do seu
   município** — se e somente se o município tiver alguma região válida. A origem
   fica registrada em `origem_regiao`, então o notebook 2 pode excluir os inferidos
   de qualquer número sensível.

In [7]:
def normaliza_regsaude(v):
    '''Numérico -> 4 dígitos. Texto livre ou vazio -> None. Nunca inventa código.'''
    if not isinstance(v, str):
        return None
    t = v.strip()
    if t == "" or not t.isdigit():
        return None
    return t.zfill(4)

cnes = cnes.copy()
cnes["regiao_saude"] = cnes.REGSAUDE.map(normaliza_regsaude)

brutos   = cnes.REGSAUDE.astype(str).str.strip()
nao_num  = brutos[(brutos != "") & (~brutos.str.isdigit())]

print(f"valores brutos distintos     : {cnes.REGSAUDE.nunique()}")
print(f"códigos válidos após normalizar: {cnes.regiao_saude.nunique()}")
print(f"linhas sem região             : {cnes.regiao_saude.isna().sum():,} "
      f"({cnes.regiao_saude.isna().mean()*100:.1f}%)")
print(f"\nrótulos de texto livre descartados ({nao_num.nunique()} distintos):")
print("  ", sorted(nao_num.unique().tolist()))

valores brutos distintos     : 82
códigos válidos após normalizar: 46
linhas sem região             : 47,743 (23.9%)

rótulos de texto livre descartados (11 distintos):
   ['DRS1', 'DRS5', 'DRS6', 'DRS7', 'GSP', 'I', 'MC', 'R17', 'R216', 'XVI', 'XVII']


In [8]:
# Região modal por município, calculada só sobre registros com código válido
reg_por_munic = (cnes.dropna(subset=["regiao_saude"])
                     .groupby("CODUFMUN").regiao_saude
                     .agg(lambda s: s.mode().iloc[0])
                     .rename("regiao_por_municipio"))

# Região declarada por hospital: a modal entre suas próprias linhas válidas
reg_por_hosp = (cnes.dropna(subset=["regiao_saude"])
                    .groupby("CNES").regiao_saude
                    .agg(lambda s: s.mode().iloc[0])
                    .rename("regiao_declarada"))

print(f"municípios com região inferível : {len(reg_por_munic)}")
print(f"estabelecimentos do CNES/LT com região declarada: {len(reg_por_hosp)}")
print("   (o recorte para os 669 hospitais do SIH acontece na dim_hospital, seção 7)")

municípios com região inferível : 343
estabelecimentos do CNES/LT com região declarada: 1064
   (o recorte para os 669 hospitais do SIH acontece na dim_hospital, seção 7)


---
## 7 · `dim_hospital`

Um registro por CNES, restrito aos **669 hospitais que aparecem no SIH** — o CNES/LT
traz 1.534 estabelecimentos, mas os que não internam ninguém no período não têm
lugar num painel de pressão hospitalar.

In [9]:
hosp_sih = sih.CNES.unique()

dim_hospital = (cnes[cnes.CNES.isin(hosp_sih)]
                .sort_values(["CNES", "_ano", "_mes"])
                .groupby("CNES")
                .agg(municipio_cod6 = ("CODUFMUN", "last"),
                     tipo_unidade   = ("TP_UNID", "last"),
                     esfera         = ("ESFERA_A", "last"),
                     natureza_jur   = ("NAT_JUR", "last"),
                     gestao         = ("TPGESTAO", "last"))
                .reset_index())

dim_hospital = (dim_hospital
                .merge(reg_por_hosp, on="CNES", how="left")
                .merge(reg_por_munic, left_on="municipio_cod6", right_index=True, how="left"))

# Cascata: região do próprio hospital → região do município → sem região
dim_hospital["regiao_saude"] = dim_hospital.regiao_declarada.fillna(dim_hospital.regiao_por_municipio)
dim_hospital["origem_regiao"] = np.select(
    [dim_hospital.regiao_declarada.notna(),
     dim_hospital.regiao_declarada.isna() & dim_hospital.regiao_por_municipio.notna()],
    ["declarada", "inferida_por_municipio"],
    default="sem_regiao",
)
dim_hospital = dim_hospital.drop(columns=["regiao_declarada", "regiao_por_municipio"])

# O código IBGE de 7 dígitos não é derivável do de 6 sem tabela de referência.
# Ele vive em dim_municipio, preenchido só quando o de-para do IBGE está disponível.
# Não criamos aqui uma coluna "cod7" com 6 dígitos: seria o join silenciosamente
# quebrado contra o qual a seção 8 adverte.

print(f"hospitais na dimensão: {len(dim_hospital)}")
print("\norigem da região de saúde:")
print(dim_hospital.origem_regiao.value_counts().to_string())
print(f"\nregiões de saúde distintas: {dim_hospital.regiao_saude.nunique()}")
print("\namostra:")
print(dim_hospital.head(5).to_string(index=False))

hospitais na dimensão: 669

origem da região de saúde:
origem_regiao
declarada                 579
inferida_por_municipio     74
sem_regiao                 16

regiões de saúde distintas: 36

amostra:
   CNES municipio_cod6 tipo_unidade esfera natureza_jur gestao regiao_saude origem_regiao
0008028         353440           05                1244      M         0201     declarada
0008036         353440           07                1244      M         0201     declarada
0008052         353440           05                1023      E         0201     declarada
0008087         353440           20                1244      M         0201     declarada
0008141         353440           20                1244      M         0201     declarada


---
## 8 · `dim_municipio`

O SIH grava `MUNIC_MOV` com **6 dígitos**; o IBGE usa **7** (o sétimo é dígito
verificador). Qualquer cruzamento com população, PIB ou malha geográfica quebra em
silêncio se isso não for tratado — a base guarda as duas formas.

O de-para de nomes vem de `dados/referencias/municipios_ibge.csv`, materializado pelo
notebook 0 a partir da API de localidades do IBGE. É ele que permite ao painel dizer
*Barueri* em vez de `350570`, e é a única fonte do código de 7 dígitos — o dígito
verificador **não é derivável** do código de 6.

In [10]:
mun_codigos = sorted(set(sih.MUNIC_MOV.unique()) | set(cnes.CODUFMUN.unique()))

dim_municipio = pd.DataFrame({"municipio_cod6": mun_codigos})
dim_municipio["uf"] = "SP"
dim_municipio = dim_municipio.merge(reg_por_munic.rename("regiao_saude"),
                                    left_on="municipio_cod6", right_index=True, how="left")

# --- de-para IBGE, se disponível -----------------------------------------
ARQ_IBGE = BASE / "dados" / "referencias" / "municipios_ibge.csv"
if ARQ_IBGE.exists():
    ref = pd.read_csv(ARQ_IBGE, dtype=str)              # espera: codigo_ibge7, nome
    ref["municipio_cod6"] = ref.codigo_ibge7.str[:6]
    dim_municipio = dim_municipio.merge(
        ref[["municipio_cod6", "codigo_ibge7", "nome"]].rename(
            columns={"codigo_ibge7": "municipio_cod7", "nome": "municipio_nome"}),
        on="municipio_cod6", how="left")
    print(f"de-para IBGE aplicado: {dim_municipio.municipio_nome.notna().sum()} de {len(dim_municipio)} nomeados")
else:
    dim_municipio["municipio_cod7"] = pd.NA
    dim_municipio["municipio_nome"] = pd.NA
    print(f"⚠  {ARQ_IBGE} não encontrado — municípios ficam sem nome.")
    print("   O painel exibirá códigos até que o de-para seja adicionado.")

print(f"\nmunicípios: {len(dim_municipio)} | com região: {dim_municipio.regiao_saude.notna().sum()}")

de-para IBGE aplicado: 364 de 364 nomeados

municípios: 364 | com região: 343


---
## 9 · `fato_internacao`

A tabela-verdade: **uma linha por AIH**, 5,2 milhões de registros, enxugada às
colunas que os índices usam e enriquecida com as chaves das dimensões e as flags de
qualidade.

As flags são o coração da decisão "flag em vez de descarte" — elas viajam com o dado
até o notebook 2, onde cada índice aplica o filtro que faz sentido para ele.

In [11]:
fato = sih.copy()

# --- Tipagem -------------------------------------------------------------
for c in ["QT_DIARIAS", "DIAS_PERM", "MORTE", "IDADE", "UTI_MES_TO"]:
    fato[c] = pd.to_numeric(fato[c], errors="coerce").fillna(0).astype("int32")
fato["VAL_TOT"] = pd.to_numeric(fato.VAL_TOT, errors="coerce").fillna(0.0)

# Datas chegam como string AAAAMMDD
fato["dt_internacao"] = pd.to_datetime(fato.DT_INTER, format="%Y%m%d", errors="coerce")
fato["dt_saida"]      = pd.to_datetime(fato.DT_SAIDA, format="%Y%m%d", errors="coerce")

# --- Chaves --------------------------------------------------------------
fato = fato.rename(columns={"MUNIC_MOV": "municipio_cod6", "DIAG_PRINC": "cid_principal"})
fato = fato.merge(dim_hospital[["CNES", "regiao_saude", "origem_regiao"]], on="CNES", how="left")
fato = fato.merge(dim_especialidade[["ESPEC", "especialidade"]], on="ESPEC", how="left")
fato = fato.merge(dim_cid[["DIAG_PRINC", "capitulo", "capitulo_desc"]].rename(
                      columns={"DIAG_PRINC": "cid_principal", "capitulo": "cid_capitulo",
                               "capitulo_desc": "cid_capitulo_desc"}),
                  on="cid_principal", how="left")

# --- Flags de qualidade --------------------------------------------------
fato["fl_sem_diaria"]     = (fato.QT_DIARIAS == 0).astype("int8")   # ~401 mil
fato["fl_sem_valor"]      = (fato.VAL_TOT == 0).astype("int8")      # ~7,8 mil
fato["fl_obito_sem_val"]  = ((fato.VAL_TOT == 0) & (fato.MORTE == 1)).astype("int8")  # 20
fato["fl_uti"]            = ((fato.MARCA_UTI != "00") | (fato.UTI_MES_TO > 0)).astype("int8")

COLS_FATO = [
    "CNES", "municipio_cod6", "MUNIC_RES", "regiao_saude", "origem_regiao",
    "_ano", "_mes", "dt_internacao", "dt_saida",
    "ESPEC", "especialidade", "cid_principal", "cid_capitulo", "cid_capitulo_desc",
    "QT_DIARIAS", "DIAS_PERM", "MORTE", "VAL_TOT", "UTI_MES_TO", "MARCA_UTI",
    "IDADE", "SEXO", "CAR_INT", "COMPLEX",
    "fl_sem_diaria", "fl_sem_valor", "fl_obito_sem_val", "fl_uti",
]
fato_internacao = fato[COLS_FATO]

print(f"fato_internacao: {fato_internacao.shape[0]:,} × {fato_internacao.shape[1]}")
print("\nflags:")
for c in ["fl_sem_diaria", "fl_sem_valor", "fl_obito_sem_val", "fl_uti"]:
    n = int(fato_internacao[c].sum())
    print(f"  {c:<18} {n:>9,}  ({n/len(fato_internacao)*100:5.2f}%)")
print(f"\nsem região de saúde: {fato_internacao.regiao_saude.isna().sum():,}")

fato_internacao: 5,210,357 × 28

flags:
  fl_sem_diaria        400,958  ( 7.70%)
  fl_sem_valor           7,827  ( 0.15%)
  fl_obito_sem_val          20  ( 0.00%)
  fl_uti               523,422  (10.05%)

sem região de saúde: 62,303


---
## 10 · `fato_leitos_mensal`

O denominador do IPH. O CNES/LT traz uma linha por **tipo de leito** — em média 6 por
hospital-mês, até 49. Somamos `QT_SUS` para chegar aos leitos SUS do hospital no mês.

`QT_SUS = 0` aparece em 89.645 linhas: são tipos de leito existentes mas não ofertados
ao SUS. Somar zeros é inofensivo; o que importa é o total por hospital.

In [12]:
fato_leitos_mensal = (cnes.groupby(["CNES", "_ano", "_mes"], as_index=False)
                          .agg(leitos_sus     = ("QT_SUS", "sum"),
                               leitos_totais  = ("QT_EXIST", "sum"),
                               tipos_de_leito = ("CODLEITO", "nunique")))

fato_leitos_mensal = fato_leitos_mensal[fato_leitos_mensal.CNES.isin(hosp_sih)]
fato_leitos_mensal = fato_leitos_mensal.merge(dim_tempo[["_ano", "_mes", "dias_no_mes"]],
                                              on=["_ano", "_mes"], how="left")

# Capacidade teórica do mês: leitos SUS × dias — denominador do IPH
fato_leitos_mensal["denominador"] = fato_leitos_mensal.leitos_sus * fato_leitos_mensal.dias_no_mes

print(f"fato_leitos_mensal: {fato_leitos_mensal.shape[0]:,} × {fato_leitos_mensal.shape[1]}")
print(f"hospitais: {fato_leitos_mensal.CNES.nunique()}")
print(f"hospital-meses sem leito SUS: {(fato_leitos_mensal.leitos_sus == 0).sum():,}")
print("\namostra:")
print(fato_leitos_mensal.head(5).to_string(index=False))

fato_leitos_mensal: 15,533 × 8
hospitais: 669
hospital-meses sem leito SUS: 77

amostra:
   CNES  _ano  _mes  leitos_sus  leitos_totais  tipos_de_leito  dias_no_mes  denominador
0008028  2022     1         211            231               9           31         6541
0008028  2022     2         211            231               9           28         5908
0008028  2022     3         201            231               9           31         6231
0008028  2022     4         201            231               9           30         6030
0008028  2022     5         201            231               9           31         6231


---
## 11 · As três bases analíticas

Cada uma existe para um conjunto de índices, no grão exato em que eles são calculados.

### 11.1 · `base_hospital_mes` → IPH e IS

Grão: **CNES × ano × mês**. Junta o paciente-dia do SIH com os leitos do CNES.
`patient_days` é o numerador do IPH; `denominador` já vem pronto da seção 10.

In [13]:
agg_hosp_mes = (fato_internacao.groupby(["CNES", "_ano", "_mes"], as_index=False)
                .agg(patient_days   = ("QT_DIARIAS", "sum"),
                     internacoes    = ("CNES", "size"),
                     dias_perm_soma = ("DIAS_PERM", "sum"),
                     obitos         = ("MORTE", "sum"),
                     valor_total    = ("VAL_TOT", "sum"),
                     internacoes_uti= ("fl_uti", "sum")))

base_hospital_mes = (agg_hosp_mes
    .merge(fato_leitos_mensal[["CNES", "_ano", "_mes", "leitos_sus", "dias_no_mes", "denominador"]],
           on=["CNES", "_ano", "_mes"], how="left")
    .merge(dim_hospital[["CNES", "municipio_cod6", "regiao_saude", "origem_regiao"]],
           on="CNES", how="left"))

base_hospital_mes["permanencia_media"] = (base_hospital_mes.patient_days
                                          / base_hospital_mes.internacoes).round(4)

print(f"base_hospital_mes: {base_hospital_mes.shape[0]:,} × {base_hospital_mes.shape[1]}")
print(f"hospital-meses sem denominador: {base_hospital_mes.denominador.isna().sum()}")
print("\namostra:")
print(base_hospital_mes.head(4).to_string(index=False))

base_hospital_mes: 14,821 × 16
hospital-meses sem denominador: 0

amostra:
   CNES  _ano  _mes  patient_days  internacoes  dias_perm_soma  obitos  valor_total  internacoes_uti  leitos_sus  dias_no_mes  denominador municipio_cod6 regiao_saude origem_regiao  permanencia_media
0008028  2022     1          4035          859            4430      99   1150793.47              115         211           31         6541         353440         0201     declarada             4.6973
0008028  2022     2          3944          778            4311      97   1257875.96               89         211           28         5908         353440         0201     declarada             5.0694
0008028  2022     3          4044          837            4332      82   1019140.80              104         201           31         6231         353440         0201     declarada             4.8315
0008028  2022     4          4100          832            4445      69   1018537.70               90         201           30

### 11.2 · `base_hospital_espec_mes` → TMH e CMI

Grão: **CNES × especialidade × ano × mês**. O CMI exclui as AIH sem valor
(`fl_sem_valor`), por isso `internacoes_com_valor` é contada em separado — dividir
o valor total por *todas* as internações subestimaria o custo médio.

In [14]:
base_hospital_espec_mes = (fato_internacao.groupby(
        ["CNES", "ESPEC", "especialidade", "_ano", "_mes"], as_index=False)
    .agg(internacoes           = ("CNES", "size"),
         obitos                = ("MORTE", "sum"),
         patient_days          = ("QT_DIARIAS", "sum"),
         valor_total           = ("VAL_TOT", "sum"),
         internacoes_com_valor = ("fl_sem_valor", lambda s: int((s == 0).sum())),
         dias_perm_soma        = ("DIAS_PERM", "sum")))

base_hospital_espec_mes = base_hospital_espec_mes.merge(
    dim_hospital[["CNES", "municipio_cod6", "regiao_saude"]], on="CNES", how="left")

print(f"base_hospital_espec_mes: {base_hospital_espec_mes.shape[0]:,} × {base_hospital_espec_mes.shape[1]}")
print("\ncontrole — TMH por especialidade em SP (agregado das duas competências):")
chk = (base_hospital_espec_mes.groupby("especialidade", as_index=False)
       .agg(internacoes=("internacoes", "sum"), obitos=("obitos", "sum")))
chk["TMH_pct"] = (chk.obitos / chk.internacoes * 100).round(2)
print(chk.sort_values("TMH_pct", ascending=False).to_string(index=False))

base_hospital_espec_mes: 43,407 × 13

controle — TMH por especialidade em SP (agregado das duas competências):
           especialidade  internacoes  obitos  TMH_pct
          Clínica médica      1805543  229781    12.73
                Crônicos        64610    2871     4.44
              Tisiologia         6619     205     3.10
                Cirurgia      1729962   37875     2.19
               Pediatria       459001    7071     1.54
Hospital-dia (cirúrgico)       267239    1059     0.40
    Outras / não mapeada        25097      29     0.12
             Psiquiatria       126383     140     0.11
            Reabilitação         1991       2     0.10
             Obstetrícia       723912     143     0.02


### 11.3 · `base_hospital_cid` → IPR

Grão: **CNES × CID-10 principal**. O IPR compara a permanência média de um hospital
num diagnóstico contra a média regional no mesmo diagnóstico.

Aqui o filtro de `QT_DIARIAS = 0` **é aplicado**, e é a única base onde isso acontece:
uma AIH sem diária não representa permanência e puxaria a média para baixo
artificialmente. A decisão está registrada em `DECISOES.md` § 2.

In [15]:
com_diaria = fato_internacao[fato_internacao.fl_sem_diaria == 0]
print(f"AIH consideradas: {len(com_diaria):,} de {len(fato_internacao):,} "
      f"(excluídas {len(fato_internacao) - len(com_diaria):,} sem diária)")

base_hospital_cid = (com_diaria.groupby(
        ["CNES", "regiao_saude", "cid_principal", "cid_capitulo"], as_index=False)
    .agg(internacoes       = ("CNES", "size"),
         patient_days      = ("QT_DIARIAS", "sum"),
         obitos            = ("MORTE", "sum"),
         valor_total       = ("VAL_TOT", "sum")))

base_hospital_cid["permanencia_media"] = (base_hospital_cid.patient_days
                                          / base_hospital_cid.internacoes).round(4)

print(f"\nbase_hospital_cid: {base_hospital_cid.shape[0]:,} × {base_hospital_cid.shape[1]}")
print(f"pares hospital×CID: {len(base_hospital_cid):,}")
print("\namostra:")
print(base_hospital_cid.head(4).to_string(index=False))

AIH consideradas: 4,809,399 de 5,210,357 (excluídas 400,958 sem diária)



base_hospital_cid: 361,273 × 9
pares hospital×CID: 361,273

amostra:
   CNES regiao_saude cid_principal cid_capitulo  internacoes  patient_days  obitos  valor_total  permanencia_media
0008028         0201          A049            I            1             4       0       626.16             4.0000
0008028         0201          A085            I            3             8       0      1248.18             2.6667
0008028         0201           A09            I           35           115       1     17389.33             3.2857
0008028         0201           A15            I           10            65       2     16720.96             6.5000


---
## 12 · Validação contra os números da Sprint 1

Antes de gravar, conferimos que as bases reproduzem os números já apresentados. O IPH
é calculado **aqui e só aqui**, como teste — a base guarda os ingredientes, não o índice.

Se estes valores não baterem, algo na engenharia divergiu e as bases **não devem ser
promovidas**.

In [16]:
v = base_hospital_mes.copy()
v["iph"] = np.where(v.denominador > 0, v.patient_days / v.denominador, np.nan)
v["classe"] = pd.cut(v.iph, [-np.inf, 0.70, 0.85, np.inf],
                     labels=["Normal", "Atenção", "Crítico"])

iph_medio  = v.iph.mean()
pct_crit   = (v.iph > 0.85).sum() / v.iph.notna().sum() * 100
pct_aten   = ((v.iph > 0.70) & (v.iph <= 0.85)).sum() / v.iph.notna().sum() * 100

checks = [
    ("linhas em base_hospital_mes",  len(v),                14821,     0),
    ("linhas SIH carregadas",        len(fato_internacao),  5210357,   0),
    ("hospitais únicos",             v.CNES.nunique(),      669,       0),
    ("municípios únicos",            sih.MUNIC_MOV.nunique(), 331,     0),
    ("IPH médio (hospital-mês)",     round(iph_medio, 4),   0.4403,    0.01),
    ("% hospital-meses Crítico",     round(pct_crit, 1),    7.8,       0.5),
    ("% hospital-meses Atenção",     round(pct_aten, 1),    10.5,      0.5),
]

print(f"{'verificação':<32} {'obtido':>12} {'esperado':>12}   ")
print("-" * 64)
ok = True
for nome, obtido, esperado, tol in checks:
    passou = abs(obtido - esperado) <= tol
    ok &= passou
    print(f"{nome:<32} {obtido:>12} {esperado:>12}   {'OK' if passou else 'DIVERGE'}")

print("\n" + ("Todas as verificações passaram." if ok else
              "ATENÇÃO: há divergência — não promover as bases."))

verificação                            obtido     esperado   
----------------------------------------------------------------
linhas em base_hospital_mes             14821        14821   OK
linhas SIH carregadas                 5210357      5210357   OK
hospitais únicos                          669          669   OK
municípios únicos                         331          331   OK
IPH médio (hospital-mês)               0.4403       0.4403   OK
% hospital-meses Crítico                  7.8          7.8   OK
% hospital-meses Atenção                 10.5         10.5   OK

Todas as verificações passaram.


---
## 13 · Gravação

Escrita em `dados/curados/`. Com `SOBRESCREVER = False`, um arquivo já existente
interrompe a gravação em vez de ser substituído — a proteção que faltava ao pipeline
original, onde um "Run All" distraído destruía os resultados bons.

In [17]:
SAIDAS = {
    "dim_tempo":               dim_tempo,
    "dim_hospital":            dim_hospital,
    "dim_municipio":           dim_municipio,
    "dim_especialidade":       dim_especialidade,
    "dim_cid":                 dim_cid,
    "fato_internacao":         fato_internacao,
    "fato_leitos_mensal":      fato_leitos_mensal,
    "base_hospital_mes":       base_hospital_mes,
    "base_hospital_espec_mes": base_hospital_espec_mes,
    "base_hospital_cid":       base_hospital_cid,
}

if not ok:
    raise RuntimeError("Validação falhou na seção 12 — gravação abortada.")

existentes = [n for n in SAIDAS if (DIR_OUT / f"{n}.parquet").exists()]
if existentes and not SOBRESCREVER:
    raise FileExistsError(
        f"Já existem: {existentes}. Defina SOBRESCREVER = True se a intenção é substituir."
    )

for nome, df in SAIDAS.items():
    caminho = DIR_OUT / f"{nome}.parquet"
    df.to_parquet(caminho, index=False)
    print(f"{nome:<26} {df.shape[0]:>9,} × {df.shape[1]:<3}  {caminho.stat().st_size/1e6:>7.1f} MB")

print(f"\ngravado em {DIR_OUT}")

dim_tempo                         24 × 8        0.0 MB


dim_hospital                     669 × 8        0.0 MB
dim_municipio                    364 × 5        0.0 MB
dim_especialidade                 16 × 5        0.0 MB
dim_cid                        9,212 × 5        0.1 MB


fato_internacao            5,210,357 × 28      75.6 MB
fato_leitos_mensal            15,533 × 8        0.0 MB
base_hospital_mes             14,821 × 16       0.4 MB
base_hospital_espec_mes       43,407 × 13       0.7 MB
base_hospital_cid            361,273 × 9        4.4 MB

gravado em <repo>/02_oracle_medflow/sprint_2_em_andamento/dados/curados


In [18]:
# Dicionário das saídas — acompanha as bases para quem for consumir no notebook 2,
# no Oracle ou no dashboard.
linhas = ["# Dicionário — bases curadas MedFlow", "",
          f"Gerado pelo notebook `01_engenharia_dados.ipynb`. Recorte: SP {ANOS[0]}–{ANOS[1]}.", ""]
for nome, df in SAIDAS.items():
    linhas += [f"## `{nome}`", "",
               f"{df.shape[0]:,} linhas × {df.shape[1]} colunas", "",
               "| coluna | tipo | nulos |", "|---|---|---:|"]
    for c in df.columns:
        linhas.append(f"| `{c}` | {df[c].dtype} | {df[c].isna().sum():,} |")
    linhas.append("")

(DIR_OUT / "DICIONARIO.md").write_text("\n".join(linhas), encoding="utf-8")
print(f"dicionário escrito em {DIR_OUT / 'DICIONARIO.md'}")

dicionário escrito em <repo>/02_oracle_medflow/sprint_2_em_andamento/dados/curados/DICIONARIO.md


---
## O que fica pronto para o notebook 2

| Índice | Base | Cálculo |
|---|---|---|
| **IPH** | `base_hospital_mes` | `patient_days ÷ denominador` |
| **IPR** | `base_hospital_cid` | `permanencia_media` do hospital ÷ média regional no mesmo CID |
| **IS** | `base_hospital_mes` | `internacoes` do mês ÷ média histórica do mesmo mês |
| **TMH** | `base_hospital_espec_mes` | `obitos ÷ internacoes × 100` |
| **CMI** | `base_hospital_espec_mes` | `valor_total ÷ internacoes_com_valor` |

### Pendências que este notebook expõe e não resolve

1. **81 hospitais sem região declarada** — herdaram a região do município. `origem_regiao`
   permite excluí-los de números sensíveis.